# 02 — Vector Store & Retrieval
**Phase:** Foundation — Day 1

**What this notebook does:**
- Reopens the ChromaDB built in notebook 01
- Explores what is stored inside
- Experiments with different queries to understand retrieval
- Visualises distance scores

**Run notebook 01 before this.**

In [1]:
import chromadb
from sentence_transformers import SentenceTransformer

# Reopen — no re-embedding needed
client = chromadb.PersistentClient(path="../chroma_db")
collection = client.get_collection("my_docs")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

print(f"Collection: {collection.name}")
print(f"Total chunks stored: {collection.count()}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Collection: my_docs
Total chunks stored: 247


## Step 1 — Peek inside the database
See what the first few stored entries look like.

In [2]:
# Peek at first 3 entries
sample = collection.peek(limit=3)

for i in range(len(sample['ids'])):
    print(f"\n{'='*50}")
    print(f"ID      : {sample['ids'][i]}")
    print(f"Text    : {sample['documents'][i][:250]}")
    print(f"Embed[0]: {sample['embeddings'][i][:5]}  ... (384 total numbers)")

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given



ID      : chunk_0
Text    : [Page 1]
24TU05MJC2 
Operating System 
L T  P  C 
Version 1.0 
 
2 1 0 3 
Pre-requisites/Exposure 
Data Structures Algorithm  
Co-requisites 
 
 
COURSE OBJECTIVES 
• To understand the fundamental concepts, architecture, and functions of modern 
oper
Embed[0]: [-0.030184263363480568, 0.02474668249487877, -0.025201115757226944, -0.03295927494764328, 0.03166845440864563]  ... (384 total numbers)

ID      : chunk_1
Text    : problem-solving related to OS design. 
COURSE OUTCOMES (COS) 
After successful completion of the course, the students will be able to: 
• Explain the structure, functions, and components of operating systems.  
• Apply process scheduling and synchron
Embed[0]: [-0.025932947173714638, 0.11781661957502365, -0.041322000324726105, -0.06366780400276184, -0.009995250031352043]  ... (384 total numbers)

ID      : chunk_10
Text    : interfering with the proper operation of the system. 
• Every computer system must have at least one operating syste

## Step 2 — Understanding distance scores
Run the same query multiple ways and compare scores.

In [3]:
def retrieve(query, top_k=5):
    vec = embed_model.encode([query]).tolist()
    res = collection.query(query_embeddings=vec, n_results=top_k)
    return res

def show_results(query):
    res = retrieve(query)
    print(f"\nQuery: '{query}'")
    for i, (doc, cid, dist) in enumerate(
        zip(res['documents'][0], res['ids'][0], res['distances'][0])
    ):
        bar = '█' * int((1 - dist) * 20) if dist < 1 else ''
        quality = 'STRONG' if dist < 0.3 else 'GOOD' if dist < 0.6 else 'WEAK'
        print(f"  {cid:12} dist={dist:.4f} {bar:20} [{quality}]")
        print(f"  {doc[:120]}...\n")

# CHANGE THESE to match your document
show_results("What is the main topic?")
show_results("Something completely unrelated like pizza recipes")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



Query: 'What is the main topic?'
  chunk_1      dist=0.7212 █████                [WEAK]
  problem-solving related to OS design. 
COURSE OUTCOMES (COS) 
After successful completion of the course, the students wi...

  chunk_7      dist=0.7250 █████                [WEAK]
  Prentice Hall of India. 
Reference Books 
1. Charles Crowley, Operating System: A Design-oriented Approach, 1st Edition,...

  chunk_13     dist=0.7550 ████                 [WEAK]
  program consists of compilers, loaders, editors, OS, etc. The application program 
consists of business programs, databa...

  chunk_31     dist=0.7634 ████                 [WEAK]
  [Page 13]
1. Batch Operating System  
In the era of 1970s, the Batch processing was very popular. The Jobs were 
execute...

  chunk_2      dist=0.7655 ████                 [WEAK]
  part of a team-based project and communicate technical findings clearly. 
 
UNIT I – Introduction to Operating Systems  ...


Query: 'Something completely unrelated like pizza recip

## Step 3 — Semantic similarity experiment
Prove that semantic search finds meaning, not just keywords.

In [4]:
# These two queries mean the same thing but use different words
# A keyword search would find different results.
# A semantic search should find SIMILAR results.

# CHANGE both queries to paraphrases of something in your document
query_a = "How does the model get trained?"      # direct phrasing
query_b = "What is the learning procedure?"      # paraphrase

res_a = retrieve(query_a)
res_b = retrieve(query_b)

ids_a = set(res_a['ids'][0])
ids_b = set(res_b['ids'][0])
overlap = ids_a & ids_b

print(f"Query A top chunks: {res_a['ids'][0]}")
print(f"Query B top chunks: {res_b['ids'][0]}")
print(f"\nOverlapping chunks: {overlap}")
print(f"Overlap count: {len(overlap)}/5")
print("\nIf overlap >= 3, semantic search is working correctly.")

Query A top chunks: ['chunk_81', 'chunk_114', 'chunk_135', 'chunk_134', 'chunk_112']
Query B top chunks: ['chunk_81', 'chunk_1', 'chunk_166', 'chunk_7', 'chunk_127']

Overlapping chunks: {'chunk_81'}
Overlap count: 1/5

If overlap >= 3, semantic search is working correctly.


## Key observations
- Distance < 0.3 = very strong match
- Distance 0.3–0.6 = good match  
- Distance > 0.6 = weak match — the answer may not be in your doc
- Semantic search finds paraphrases — keyword search cannot do this

**Next:** `03_groq_rag_pipeline.ipynb`